# Regularization: Ridge vs. Lasso 🥊🥊
Ridge and Lasso are two variations of regularization. The difference comes from a slight change in the cost function used to train the model. In this exercise, we compare these two strategies and analyze their influence on the model's coefficients, and the effect of the regularization strength $\alpha$.

## The dataset 🧬
`data/gene_expression_regression.csv` contains the expression level of 2,000 genes measured in 801 tumour samples (from the public *gene expression cancer RNA-Seq* dataset of the UCI Machine Learning Repository). The `target` column is a score built for this course from **20 of these genes**, plus some noise. Which ones? That's the question: `data/gene_expression_true_genes.csv` gives the answer, so at the end we'll check whether the models found them.

## Importing libraries and loading the dataset
1. Import the usual libraries. Don't forget to import scikit-learn's following model classes:
* LinearRegression
* Ridge
* Lasso

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

import plotly.io as pio
import plotly.graph_objects as go
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

2. Load the dataset.

In [ ]:
dataset = pd.read_csv("data/gene_expression_regression.csv")
dataset.head()

,gene_6950,gene_4781,gene_3013,gene_1339,gene_7012,gene_1744,gene_1428,gene_13446,gene_15688,gene_10542,...,gene_3552,gene_336,gene_1526,gene_832,gene_6246,gene_5476,gene_10379,gene_12930,gene_17491,target
0,0.0000,10.0608,5.3419,6.9332,10.2884,7.0918,9.9521,1.3343,10.7109,11.0690,...,3.9260,7.2825,11.3939,9.1754,10.0035,5.7165,11.8603,9.3988,11.2720,58.0314
1,0.3237,9.7803,3.5626,9.3508,10.6184,9.1127,9.3814,0.0000,8.6012,10.8085,...,0.0000,2.2549,10.8169,7.4249,7.0476,6.9964,11.3363,7.6829,11.7615,49.9721
2,0.4526,10.1151,1.8398,0.0000,11.1579,9.7775,9.4895,0.7966,10.0934,9.2789,...,2.2280,1.6830,8.7639,9.2178,6.8530,8.4070,11.2833,7.8352,11.5131,77.7376
3,1.2674,9.5233,5.1191,5.3612,11.2572,9.1437,9.4152,1.6372,11.1849,10.1669,...,1.2674,6.0690,10.8005,8.9356,7.9333,8.0946,12.3795,7.7916,11.5852,59.2280
4,0.8897,10.0400,4.6369,0.0000,10.7868,9.1113,8.8852,0.6494,9.6239,10.1774,...,1.2758,6.5006,10.4511,8.2219,7.5809,8.6610,11.2833,8.4100,11.5575,48.5670


## Exploration
3. How many examples and how many features are there? Are there missing values? Why could this dataset be a problem for a linear regression?

In [ ]:
print("Number of rows :", dataset.shape[0])
print("Number of columns :", dataset.shape[1])
print("Missing values in the whole dataset:", dataset.isnull().any().any())

Number of rows : 801
Number of columns : 2001
Missing values in the whole dataset: False


**There are more features (2,000 genes) than examples (801 samples): it's very common in genetics. With more features than examples, a linear regression can fit the training data perfectly... and learn the noise.**

## Preparing the data for machine learning
4. Separate the target from the features, and make a train/test splitting (20% for the test set).

In [ ]:
target_variable = "target"
X = dataset.drop(target_variable, axis=1)
Y = dataset.loc[:, target_variable]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("train:", X_train.shape, " test:", X_test.shape)

train: (640, 2000)  test: (161, 2000)


5. What preprocessing is necessary here? Apply it to the features.

In [ ]:
# All the features are numeric, with no missing values: we only standardize them
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)  # don't fit again!

## A model without regularization
6. Train a `LinearRegression` and display its $R^2$ score on the train and test sets. What do you notice?

In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("R2 score on training set : ", round(regressor.score(X_train, Y_train), 3))
print("R2 score on test set :     ", round(regressor.score(X_test, Y_test), 3))

R2 score on training set :  1.0
R2 score on test set :      0.64


**A perfect score on the training set, a much lower one on the test set: the model overfits. With 2,000 coefficients for 640 training examples, it can fit any training data, noise included.**

## Ridge
Let's train 3 Ridge regressors with different values of the strength $\alpha$, and analyze their performances and their coefficients.

7. Declare an instance of the Ridge class with $\alpha = 1$, save it into an object called `ridge1`. Train the model and display its $R^2$ score on the train and test sets.

In [ ]:
ridge1 = Ridge(alpha=1)
ridge1.fit(X_train, Y_train)
print("R2 score on training set : ", round(ridge1.score(X_train, Y_train), 3))
print("R2 score on test set :     ", round(ridge1.score(X_test, Y_test), 3))

R2 score on training set :  1.0
R2 score on test set :      0.641


8. Same thing with $\alpha = 100$, in an object called `ridge2`.

In [ ]:
ridge2 = Ridge(alpha=100)
ridge2.fit(X_train, Y_train)
print("R2 score on training set : ", round(ridge2.score(X_train, Y_train), 3))
print("R2 score on test set :     ", round(ridge2.score(X_test, Y_test), 3))

R2 score on training set :  0.991
R2 score on test set :      0.666


9. Same thing with $\alpha = 10000$, in an object called `ridge3`.

In [ ]:
ridge3 = Ridge(alpha=10000)
ridge3.fit(X_train, Y_train)
print("R2 score on training set : ", round(ridge3.score(X_train, Y_train), 3))
print("R2 score on test set :     ", round(ridge3.score(X_test, Y_test), 3))

R2 score on training set :  0.627
R2 score on test set :      0.438


**When $\alpha$ increases, the training score decreases: the penalty prevents the model from fitting the training data perfectly (it increases the bias). The test score first improves a little ($\alpha = 100$), then drops when the penalty becomes too strong ($\alpha = 10000$): the model underfits.**

11. Store the coefficients of the three Ridge models in a DataFrame, and plot them in the same figure. What do you notice?

In [ ]:
coefficients_ridge = pd.DataFrame({
    "Feature": X.columns,
    "Ridge1": ridge1.coef_,
    "Ridge2": ridge2.coef_,
    "Ridge3": ridge3.coef_,
})
fig = px.line(coefficients_ridge, x="Feature", y=["Ridge1", "Ridge2", "Ridge3"], title="Ridge coefficients")
fig.update_xaxes(showticklabels=False)
fig.show()

**With Ridge, when $\alpha$ increases, all the coefficients shrink towards zero, but none of them is exactly zero.**

## Lasso
Let's make the same study with Lasso.

12. Train three Lasso models with $\alpha = 0.01$, $0.1$ and $1$ (`lasso1`, `lasso2`, `lasso3`). For each one, display the $R^2$ scores and the number of coefficients that are not zero.

In [ ]:
for name, alpha in [("lasso1", 0.01), ("lasso2", 0.1), ("lasso3", 1)]:
    model = Lasso(alpha=alpha, max_iter=20000)
    model.fit(X_train, Y_train)
    globals()[name] = model  # keep the three models for the next questions
    print(f"{name} (alpha = {alpha}): train R2 {model.score(X_train, Y_train):.3f}, test R2 {model.score(X_test, Y_test):.3f}, "
          f"non-zero coefficients: {(model.coef_ != 0).sum()}")

lasso1 (alpha = 0.01): train R2 0.997, test R2 0.836, non-zero coefficients: 526
lasso2 (alpha = 0.1): train R2 0.958, test R2 0.922, non-zero coefficients: 123


lasso3 (alpha = 1): train R2 0.686, test R2 0.596, non-zero coefficients: 30


13. Plot the coefficients of the three Lasso models in the same figure. What do you notice?

In [ ]:
coefficients_lasso = pd.DataFrame({
    "Feature": X.columns,
    "Lasso1": lasso1.coef_,
    "Lasso2": lasso2.coef_,
    "Lasso3": lasso3.coef_,
})
fig = px.line(coefficients_lasso, x="Feature", y=["Lasso1", "Lasso2", "Lasso3"], title="Lasso coefficients")
fig.update_xaxes(showticklabels=False)
fig.show()

**Contrary to Ridge, Lasso sets many coefficients exactly to zero. The higher $\alpha$ is, the more coefficients are cancelled: about 500 genes are kept with $\alpha = 0.01$, about 120 with $\alpha = 0.1$, and only about 30 with $\alpha = 1$. And the test score is much better than with Ridge for $\alpha = 0.1$.**

## Hyperparameter optimization
14. Use a grid search (5 folds) to find the best value of $\alpha$, for Ridge and then for Lasso. You can test the following values:
* Ridge: $\alpha$ = `np.logspace(0, 5, 11)` (from 1 to 100,000)
* Lasso: $\alpha$ = `[0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1]`

In [ ]:
best_ridge = GridSearchCV(Ridge(), param_grid={"alpha": np.logspace(0, 5, 11)}, cv=5)
best_ridge.fit(X_train, Y_train)
print("Ridge, best hyperparameters:", best_ridge.best_params_, " best cross-validated R2:", round(best_ridge.best_score_, 3))

best_lasso = GridSearchCV(Lasso(max_iter=20000), param_grid={"alpha": [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1]}, cv=5, n_jobs=-1)
best_lasso.fit(X_train, Y_train)
print("Lasso, best hyperparameters:", best_lasso.best_params_, " best cross-validated R2:", round(best_lasso.best_score_, 3))

Ridge, best hyperparameters: {'alpha': np.float64(100.0)}  best cross-validated R2: 0.695


Lasso, best hyperparameters: {'alpha': 0.1}  best cross-validated R2: 0.904


### Comparing models
15. Display the scores on the train and test sets for the best Ridge and the best Lasso models. Which one would you choose, and why?

In [ ]:
print("RIDGE / R2 score on training set : ", round(best_ridge.score(X_train, Y_train), 3))
print("RIDGE / R2 score on test set :     ", round(best_ridge.score(X_test, Y_test), 3))
print()
print("LASSO / R2 score on training set : ", round(best_lasso.score(X_train, Y_train), 3))
print("LASSO / R2 score on test set :     ", round(best_lasso.score(X_test, Y_test), 3))

RIDGE / R2 score on training set :  0.991
RIDGE / R2 score on test set :      0.666

LASSO / R2 score on training set :  0.958
LASSO / R2 score on test set :      0.922


**Lasso is much better here ($R^2 \approx 0.92$ on the test set, against about 0.67 for Ridge). This is expected: the target depends on only a few genes, and Lasso is designed to find a few important features among many. It is also lighter: it uses about 120 genes instead of 2,000.**

## Using Lasso as an automated feature selection method

16. Extract the names of the genes that have a non-zero coefficient in the best Lasso model. How many of the 20 genes listed in `data/gene_expression_true_genes.csv` are among them?

In [ ]:
coefficients = pd.Series(best_lasso.best_estimator_.coef_, index=X.columns)
best_features = coefficients[coefficients != 0].index.to_list()
print("Genes kept by Lasso:", len(best_features))

true_genes = pd.read_csv("data/gene_expression_true_genes.csv")
found = true_genes["gene"].isin(best_features)
print(f"True genes found by Lasso: {found.sum()} out of {len(true_genes)}")

# the largest coefficients of Lasso: are they the true genes?
top = coefficients.abs().sort_values(ascending=False).head(25)
pd.DataFrame({"abs. coefficient": top.round(3), "true gene": top.index.isin(true_genes["gene"])})

Genes kept by Lasso: 123
True genes found by Lasso: 20 out of 20


,abs. coefficient,true gene
gene_10664,2.373,True
gene_5173,2.191,True
gene_18682,2.182,True
gene_14163,1.986,True
gene_4445,1.982,True
gene_12617,1.936,True
gene_7160,1.903,True
gene_11279,1.727,True
gene_892,1.553,True
gene_2741,1.533,True


**Lasso kept all the 20 genes that really drive the target, and they have the largest coefficients. The other genes it kept have small coefficients: they explain a little of the noise.**

17. Keep only the genes selected by Lasso, make a new train/test splitting, standardize, and train a NON-regularized linear regression. What do you think of the result?

In [ ]:
X_best = X.loc[:, best_features]
X_train, X_test, Y_train, Y_test = train_test_split(X_best, Y, test_size=0.2, random_state=0)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("R2 score on training set : ", round(regressor.score(X_train, Y_train), 3))
print("R2 score on test set :     ", round(regressor.score(X_test, Y_test), 3))

R2 score on training set :  0.971
R2 score on test set :      0.904


**With the genes selected by Lasso, even a non-regularized linear regression generalizes well: from $R^2 \approx 0.64$ with the 2,000 genes to about 0.90 on the test set, with about 120 genes.** 🥳

## Conclusion
Ridge and Lasso both constrain the values of the model's coefficients. When the regularization strength $\alpha$ increases, the constraint becomes stronger and the model's bias increases.

With Ridge, all the coefficients shrink, while with Lasso, some coefficients become exactly zero: Lasso can be used as an **automated feature selection** method. When only a few features matter among many, as here, Lasso is the right choice.